In [35]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from src.logger_config import setup_logging
from src.config import FINAL_DATA_PATH

logger = setup_logging()

df = pd.read_csv(FINAL_DATA_PATH)

In [ ]:
year_agg=df.groupby('year')['price'].median().reset_index()
year_agg=year_agg[(year_agg['year']<2022)]
logger.info(f"Yearly median price:\n{year_agg}")

In [ ]:
X = df[['year', 'odometer']] 
y = df['price']

# Разделяем на обучение и тест (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

# Проверяем точность
score = model.score(X_test, y_test)
logger.info(f"Baseline Linear Regression R^2 score: {score:.4f}")

2026-05-04 12:42:15 | INFO     | __main__:<module>:12 - Baseline Linear Regression R^2 score: 0.3974


In [ ]:

toyota_df = df[(df['manufacturer'] == 'toyota') & (df['year'] < 2022)]

X = toyota_df[['year']]
y = toyota_df['price']

model_toyota = LinearRegression()
model_toyota.fit(X, y)

future_car = np.array([[2028]])

predicted_price = model_toyota.predict(future_car)
logger.info(f"Predicted price for a 2028 Toyota: ${predicted_price[0]:.2f}")


2026-05-04 12:40:58 | INFO     | __main__:<module>:12 - Predicted price for a 2028 Toyota: $44889.93


c:\Projects\PythonLlmMinicondaV1\.conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [ ]:
# n_estimators=100 — количество деревьев в "лесу"
# random_state=42 — для воспроизводимости результатов
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X, y)

predicted_price = model_rf.predict(future_car)
logger.info(f"RandomForest predicted price for a 2028 Toyota: ${predicted_price[0]:.2f}")

2026-05-04 12:41:02 | INFO     | __main__:<module>:9 - RandomForest predicted price for a 2028 Toyota: $42691.68


c:\Projects\PythonLlmMinicondaV1\.conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [36]:
model_filename = 'toyota_rf_model.pkl'
joblib.dump(model_rf, model_filename)
if not os.path.exists('MLmodels'):
    os.makedirs('MLmodels')

joblib.dump(model_rf, 'MLmodels/toyota_rf_v1.pkl')

logger.info(f"Модель успешно сохранена в файл: {model_filename}")

2026-05-04 12:56:50 | INFO     | __main__:<module>:8 - Модель успешно сохранена в файл: toyota_rf_model.pkl
